<a href="https://colab.research.google.com/github/benjbastine/CSST-102-Machine-Learning-Final-Project/blob/main/CSST102_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2: Dataset Preparation and Feature Engineering

In [ ]:
import pandas as pd
import numpy as np

# Load all four label files + weekly features
rbb94 = pd.read_csv("RBB_P94.8_binary_labels.csv")
rbb75 = pd.read_csv("RBB_P75.1_binary_labels.csv")
wsb94 = pd.read_csv("WSB_P94.3_binary_labels.csv")
wsb98 = pd.read_csv("WSB_P98.9_binary_labels.csv")
weekly = pd.read_csv("weekly_lot_a_with_moon_dataset.csv")

print("=== SHAPES ===")
print("weekly:", weekly.shape)
for n, d in [("RBB_P94.8", rbb94), ("RBB_P75.1", rbb75),
             ("WSB_P94.3", wsb94), ("WSB_P98.9", wsb98)]:
    print(f"{n}: {d.shape}")

print("\n=== CLASS COUNTS (Binary_Label) ===")
for name, df in [("RBB_P75.1", rbb75), ("RBB_P94.8", rbb94),
                 ("WSB_P94.3", wsb94), ("WSB_P98.9", wsb98)]:
    counts = df['Binary_Label'].value_counts().sort_index().to_dict()
    print(f"{name}: {counts}  | positives = {counts.get(1, 0)}")

## Attempt 2 — Composite Key Merge (FAILED — expected)

**What was tried:** Merge on `(Year, Week_Number)` with `validate='one_to_one'`.

**Result:** `MergeError: Merge keys are not unique in either left or right dataset`.

**Why this is useful:** The `validate` safeguard correctly refused to inflate the data.
Duplicate (Year, Week_Number) pairs exist in both files (boundary/multi-observation weeks).
Diagnosed in **Cell** below and resolved via **verified positional merge** in the next **Cell**.


In [ ]:
# Why did composite key fail?
print("=== Duplicate (Year, Week_Number) counts ===")
for name, df in [("RBB", rbb), ("WSB", wsb), ("Weekly", weekly)]:
    dups = df.duplicated(subset=['Year','Week_Number'], keep=False).sum()
    print(f"{name}: {dups} rows share a duplicate key")

print("\n=== Which keys repeat in Weekly ===")
dup_keys = weekly[weekly.duplicated(subset=['Year','Week_Number'], keep=False)]
print(dup_keys[['Year','Week_Number']].value_counts().head(10))

print("\n=== Positional alignment verification ===")
print("RBB Week_Number == Weekly:", (rbb['Week_Number'].values == weekly['Week_Number'].values).all())
print("WSB Week_Number == Weekly:", (wsb['Week_Number'].values == weekly['Week_Number'].values).all())
print("RBB Pest_Count == Weekly RBB:", (rbb['Pest_Count'].values == weekly['RBB'].values).all())
print("WSB Pest_Count == Weekly WSB:", (wsb['Pest_Count'].values == weekly['WSB'].values).all())

In [ ]:
# FIX: positional merge
feature_cols = ['T2M', 'T2M_MIN', 'T2M_MAX', 'RH2M', 'PRECTOTCORR',
                'WS2M', 'WS2M_MAX', 'WS2M_MIN', 'WD2M', 'GWETTOP', 'Moon_Category']

rbb_merged = rbb.copy()
wsb_merged = wsb.copy()

for col in feature_cols:
    rbb_merged[col] = weekly[col].values
    wsb_merged[col] = weekly[col].values

print("RBB merged shape:", rbb_merged.shape, "| expected (366, 25)")
print("WSB merged shape:", wsb_merged.shape, "| expected (366, 25)")
print("RBB missing T2M:", rbb_merged['T2M'].isna().sum())
print("WSB missing T2M:", wsb_merged['T2M'].isna().sum())
print("\nRBB Binary_Label counts:", rbb_merged['Binary_Label'].value_counts().sort_index().to_dict())
print("WSB Binary_Label counts:", wsb_merged['Binary_Label'].value_counts().sort_index().to_dict())

In [ ]:
for name, df in [("RBB_P94.8", rbb_merged), ("WSB_P94.3", wsb_merged)]:
    print("\n" + "="*55)
    print(f"  {name}")
    print("="*55)
    print("Shape:", df.shape)

    print("\n--- Data Types ---")
    print(df.dtypes)

    print("\n--- Missing Values per Column ---")
    print(df.isna().sum())

    print("\n--- Duplicate (Year, Week_Number) rows ---")
    print(df.duplicated(subset=['Year', 'Week_Number']).sum())

    print("\n--- Duplicate (all columns) rows ---")
    print(df.duplicated().sum())

    print("\n--- Binary_Label Counts ---")
    print(df['Binary_Label'].value_counts().sort_index())

    print("\n--- Binary_Label Proportions ---")
    print(df['Binary_Label'].value_counts(normalize=True).round(3))

    print("\n--- Environmental Feature Ranges ---")
    print(df[['T2M','RH2M','PRECTOTCORR','WS2M','GWETTOP']].describe().round(2))

    print("\n--- Moon_Category unique values ---")
    print(sorted(df['Moon_Category'].dropna().unique()))

## Week 2 — Preprocessing Plan & Decisions

### Merge Decision: MERGE NEEDED → RESOLVED via Verified Positional Merge
- **Attempt 1 (failed):** Merged on `Week_Number` alone → 2564 rows.
  Cause: `Week_Number` is week-of-year (1–53) and repeats ~7× across years → many-to-many join.
- **Attempt 2 (failed):** Merged on composite `(Year, Week_Number)` → `MergeError`.
  Cause: duplicate (Year, Week_Number) pairs exist in both binary files and weekly dataset
  (boundary weeks / multi-observation weeks).
- **Resolution:** Verified positional merge. Four independent checks confirmed alignment:
  (a) Week_Number matches positionally across all 366 rows,
  (b) Pest_Count in binary files matches RBB/WSB columns in weekly,
  (c) all files have 366 rows,
  (d) same source order (single Zenodo dataset).
- **Result:** 366 rows per pest, 0 missing features, labels intact.
- **Note:** Duplicate (Year, Week_Number) pairs recorded in Cell A diagnostics for Week 3 investigation.

### Dataset Selection
- **RBB:** `RBB_P94.8_binary_labels.csv` (~19 positives) ✅
- **WSB:** `WSB_P94.3_binary_labels.csv` (~21 positives) ✅
- **Rejected:** `WSB_P98.9` (~4 positives) — too few for reliable classification at n=366.

### Target Variable
`Binary_Label` — 0 = background, 1 = elevated outbreak (per pest)

### Candidate Input Features (12)
- Meteorological: `T2M`, `T2M_MIN`, `T2M_MAX`, `RH2M`, `PRECTOTCORR`,
  `WS2M`, `WS2M_MAX`, `WS2M_MIN`, `WD2M`, `GWETTOP`
- Temporal: `Week_Number` (or season)
- Lunar: `Moon_Category` → **cyclic encoding** (sin/cos) to preserve phase wrap-around

### Excluded — Data Leakage (derived from target)
- `Pest_Count`, `Threshold_Count`, `Percentile`, `Percentile_Label`
- `Binary_Label_Definition`, `Comparison_Rule`
- `Observation_ID`, `Source_Row_ID` (identifiers only)

### Cleaning Plan
- Missing values: median imputation for numerics (only if any remain after merge — currently 0)
- Duplicates: verified 0 → no action needed
- Data types: `Date` → convert to `datetime64` (currently string format mismatch across files)

### Encoding Plan
- `Moon_Category` → cyclic: `moon_sin = sin(2π × phase / N)`, `moon_cos = cos(2π × phase / N)`
- `WD2M` (wind direction) → cyclic encoding (0–360°) if used

### Train/Test Split Plan
- **Primary:** Stratified 80/20 split per pest (`random_state=42`) — preserves class ratio
- **Secondary (honest check):** Time-based split (train on earlier years, test on later years)
- **Cross-validation:** Stratified 5-fold CV for model comparison (essential with n=366)
- **Acknowledged limitation:** Weekly data is autocorrelated → random split is optimistic.
  Both split results will be reported in Week 4–5.

### Class Imbalance Plan
- Baseline: `class_weight='balanced'` in Logistic Regression
- If needed: SMOTE applied **only inside training folds** (never on the full dataset)
- Evaluation: **Recall, F1, ROC-AUC, Precision** — accuracy alone is misleading

### Known Limitations (for report)
- n=366, single lot, single location (Midsayap) → no generalizability beyond this lot
- Weather variables are gridded reanalysis (NASA POWER-style), not on-site
- Observational data → associations only, no causal claims
- Temporal autocorrelation → random split overestimates performance

# Week 3: Initial Machine Learning Model Development
**Target:** Binary_Label (0 = background, 1 = elevated outbreak)

**Models:** Logistic Regression baseline (class_weight='balanced')

In [ ]:
import pandas as pd
import numpy as np

rbb = pd.read_csv("RBB_P94.8_binary_labels.csv")
wsb = pd.read_csv("WSB_P94.3_binary_labels.csv")
weekly = pd.read_csv("weekly_lot_a_with_moon_dataset.csv")

feature_cols_raw = ['T2M','T2M_MIN','T2M_MAX','RH2M','PRECTOTCORR',
                    'WS2M','WS2M_MAX','WS2M_MIN','WD2M','GWETTOP','Moon_Category']

# Verify alignment
assert (rbb['Week_Number'].values == weekly['Week_Number'].values).all()
assert (wsb['Week_Number'].values == weekly['Week_Number'].values).all()
assert (rbb['Pest_Count'].values == weekly['RBB'].values).all()
assert (wsb['Pest_Count'].values == weekly['WSB'].values).all()

rbb_merged = rbb.copy()
wsb_merged = wsb.copy()
for col in feature_cols_raw:
    rbb_merged[col] = weekly[col].values
    wsb_merged[col] = weekly[col].values

print("RBB:", rbb_merged.shape, "| WSB:", wsb_merged.shape)
print("RBB classes:", rbb_merged['Binary_Label'].value_counts().to_dict())
print("WSB classes:", wsb_merged['Binary_Label'].value_counts().to_dict())

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

def build_features(df):
    """Return X (features), y (target), and feature names."""
    out = df.copy()

    # --- Missing values: median imputation for numeric ---
    num_cols = ['T2M','T2M_MIN','T2M_MAX','RH2M','PRECTOTCORR',
                'WS2M','WS2M_MAX','WS2M_MIN','GWETTOP']
    for c in num_cols:
        out[c] = out[c].fillna(out[c].median())

    # --- Cyclic encoding: Moon_Category ---
    moon_cats = sorted(out['Moon_Category'].dropna().unique())
    n_moon = len(moon_cats)
    moon_map = {c: i for i, c in enumerate(moon_cats)}
    out['moon_idx'] = out['Moon_Category'].map(moon_map).fillna(0)
    out['moon_sin'] = np.sin(2 * np.pi * out['moon_idx'] / n_moon)
    out['moon_cos'] = np.cos(2 * np.pi * out['moon_idx'] / n_moon)

    # --- Cyclic encoding: WD2M (wind direction 0-360) ---
    wd = out['WD2M'].fillna(out['WD2M'].median())
    out['wd_sin'] = np.sin(2 * np.pi * wd / 360)
    out['wd_cos'] = np.cos(2 * np.pi * wd / 360)

    # --- Feature list ---
    features = ['T2M','T2M_MIN','T2M_MAX','RH2M','PRECTOTCORR',
                'WS2M','WS2M_MAX','WS2M_MIN','GWETTOP',
                'moon_sin','moon_cos','wd_sin','wd_cos']
    X = out[features].values
    y = out['Binary_Label'].values
    return X, y, features

X_rbb, y_rbb, feat_names = build_features(rbb_merged)
X_wsb, y_wsb, _           = build_features(wsb_merged)

print("Feature matrix RBB:", X_rbb.shape, "| WSB:", X_wsb.shape)
print("Features used:", feat_names)
print("Any NaN in X_rbb?", np.isnan(X_rbb).any())
print("Any NaN in X_wsb?", np.isnan(X_wsb).any())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
import pickle

def train_baseline(X, y, pest_name):
    # Stratified 80/20 split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42)

    # Scale features (fit on train only — prevents leakage)
    scaler = StandardScaler().fit(X_tr)
    X_tr_s = scaler.transform(X_tr)
    X_te_s = scaler.transform(X_te)

    # Baseline model
    model = LogisticRegression(class_weight='balanced', max_iter=1000,
                               random_state=42)
    model.fit(X_tr_s, y_tr)

    # Predictions
    y_pred = model.predict(X_te_s)
    y_prob = model.predict_proba(X_te_s)[:, 1]

    # Metrics
    metrics = {
        'accuracy':  accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred, zero_division=0),
        'recall':    recall_score(y_te, y_pred, zero_division=0),
        'f1':        f1_score(y_te, y_pred, zero_division=0),
        'roc_auc':   roc_auc_score(y_te, y_prob) if len(set(y_te)) > 1 else float('nan'),
    }
    cm = confusion_matrix(y_te, y_pred, labels=[0, 1])

    print(f"\n{'='*50}\n  {pest_name} — Logistic Regression Baseline\n{'='*50}")
    print(f"Train size: {len(y_tr)} | Test size: {len(y_te)}")
    print(f"Train positives: {int(y_tr.sum())} | Test positives: {int(y_te.sum())}")
    print(f"\nMetrics:")
    for k, v in metrics.items():
        print(f"  {k:10s}: {v:.4f}")
    print(f"\nConfusion matrix (rows=true, cols=pred):")
    print(cm)
    print(f"\nClassification report:")
    print(classification_report(y_te, y_pred, zero_division=0))

    # Save predictions
    pred_df = pd.DataFrame({
        'y_true': y_te, 'y_pred': y_pred, 'y_prob': y_prob
    })
    pred_df.to_csv(f"pred_{pest_name}_baseline.csv", index=False)

    return model, scaler, metrics, cm, y_te, y_pred, y_prob

results = {}
results['RBB'] = train_baseline(X_rbb, y_rbb, 'RBB')
results['WSB'] = train_baseline(X_wsb, y_wsb, 'WSB')

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    pest: results[pest][2] for pest in ['RBB','WSB']
}).T.round(4)
print("\n=== BASELINE MODEL COMPARISON ===")
print(comparison)

# Coefficient inspection (which features drove predictions?)
for pest in ['RBB','WSB']:
    model, scaler, metrics, cm, _, _, _ = results[pest]
    coefs = pd.Series(model.coef_[0], index=feat_names).sort_values(key=abs, ascending=False)
    print(f"\n--- {pest} — Top coefficients ---")
    print(coefs.round(3).head(8))

# Save comparison table
comparison.to_csv("baseline_comparison.csv")
print("\nSaved: baseline_comparison.csv")
print("Saved: pred_RBB_baseline.csv, pred_WSB_baseline.csv")

## Week 3 — Baseline Model Summary

### Preprocessing
- Verified positional merge reused from Week 2.
- Median imputation for numeric features.
- Cyclic encoding: Moon_Category (sin/cos), WD2M wind direction (sin/cos).
- StandardScaler fit on training set only — prevents leakage.
- Final feature set (13): T2M, T2M_MIN, T2M_MAX, RH2M, PRECTOTCORR,
  WS2M, WS2M_MAX, WS2M_MIN, GWETTOP, moon_sin, moon_cos, wd_sin, wd_cos.

### Split
- Stratified 80/20 train/test split (random_state=42) per pest.

### Baseline model
- Logistic Regression with `class_weight='balanced'` to handle imbalance.

### Results
| Pest | Accuracy | Precision | Recall | F1 | ROC-AUC |
|------|----------|-----------|--------|----|---------|
| RBB  | 0.7838      | 0.125       | 0.50    | 0.2000| 0.8429  |
| WSB  | 0.7027      | 0.125       | 0.75    | 0.2143| 0.7857  |
               

# Week 4: Model Evaluation and Performance Comparison
**Baseline:** Logistic Regression (Week 3)

**Second model:** Random Forest with class_weight='balanced'

**Validation:** Stratified 5-fold CV + held-out test

**Extra check:** Time-based split (train on earlier years, test on later years)

In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)

rbb = pd.read_csv("RBB_P94.8_binary_labels.csv")
wsb = pd.read_csv("WSB_P94.3_binary_labels.csv")
weekly = pd.read_csv("weekly_lot_a_with_moon_dataset.csv")

feature_cols_raw = ['T2M','T2M_MIN','T2M_MAX','RH2M','PRECTOTCORR',
                    'WS2M','WS2M_MAX','WS2M_MIN','WD2M','GWETTOP','Moon_Category']

rbb_merged = rbb.copy(); wsb_merged = wsb.copy()
for c in feature_cols_raw:
    rbb_merged[c] = weekly[c].values
    wsb_merged[c] = weekly[c].values

def build_features(df):
    out = df.copy()
    num_cols = ['T2M','T2M_MIN','T2M_MAX','RH2M','PRECTOTCORR',
                'WS2M','WS2M_MAX','WS2M_MIN','GWETTOP']
    for c in num_cols:
        out[c] = out[c].fillna(out[c].median())
    cats = sorted(out['Moon_Category'].dropna().unique())
    n = len(cats)
    out['moon_idx'] = out['Moon_Category'].map({c:i for i,c in enumerate(cats)}).fillna(0)
    out['moon_sin'] = np.sin(2*np.pi*out['moon_idx']/n)
    out['moon_cos'] = np.cos(2*np.pi*out['moon_idx']/n)
    wd = out['WD2M'].fillna(out['WD2M'].median())
    out['wd_sin'] = np.sin(2*np.pi*wd/360)
    out['wd_cos'] = np.cos(2*np.pi*wd/360)
    feats = ['T2M','T2M_MIN','T2M_MAX','RH2M','PRECTOTCORR',
             'WS2M','WS2M_MAX','WS2M_MIN','GWETTOP',
             'moon_sin','moon_cos','wd_sin','wd_cos']
    return out[feats].values, out['Binary_Label'].values, feats, out['Year'].values

X_rbb, y_rbb, feat_names, years_rbb = build_features(rbb_merged)
X_wsb, y_wsb, _, years_wsb = build_features(wsb_merged)

print("X_rbb:", X_rbb.shape, "| positives:", int(y_rbb.sum()))
print("X_wsb:", X_wsb.shape, "| positives:", int(y_wsb.sum()))

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced',
                                             max_iter=1000, random_state=42),
    'RandomForest':       RandomForestClassifier(n_estimators=300,
                                                 class_weight='balanced',
                                                 random_state=42,
                                                 min_samples_leaf=3)
}
scoring = ['accuracy','precision','recall','f1','roc_auc']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for pest, X, y in [('RBB', X_rbb, y_rbb), ('WSB', X_wsb, y_wsb)]:
    print(f"\n{'='*55}\n  {pest} — Stratified 5-Fold CV\n{'='*55}")
    for name, model in models.items():
        # Scale within each fold using a pipeline (avoids leakage)
        from sklearn.pipeline import Pipeline
        pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
        scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
        row = {m: scores[f'test_{m}'].mean() for m in scoring}
        row.update({f"{m}_std": scores[f'test_{m}'].std() for m in scoring})
        cv_results[(pest, name)] = row
        print(f"\n  {name}:")
        for m in scoring:
            print(f"    {m:10s} = {row[m]:.3f}  (± {row[f'{m}_std']:.3f})")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
test_metrics = {}

for ax, (pest, X, y) in zip(axes, [('RBB', X_rbb, y_rbb), ('WSB', X_wsb, y_wsb)]):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                              stratify=y, random_state=42)
    scaler = StandardScaler().fit(X_tr)
    X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

    print(f"\n{'='*55}\n  {pest} — Held-out test (80/20 stratified)\n{'='*55}")
    print(f"  Train: {len(y_tr)} (pos={int(y_tr.sum())}) | "
          f"Test: {len(y_te)} (pos={int(y_te.sum())})")

    for name, model in models.items():
        model.fit(X_tr_s, y_tr)
        y_pred = model.predict(X_te_s)
        y_prob = model.predict_proba(X_te_s)[:,1]
        mets = {
            'accuracy':  accuracy_score(y_te, y_pred),
            'precision': precision_score(y_te, y_pred, zero_division=0),
            'recall':    recall_score(y_te, y_pred, zero_division=0),
            'f1':        f1_score(y_te, y_pred, zero_division=0),
            'roc_auc':   roc_auc_score(y_te, y_prob) if len(set(y_te))>1 else float('nan')
        }
        test_metrics[(pest, name)] = mets
        print(f"\n  {name}:")
        for k, v in mets.items(): print(f"    {k:10s} = {v:.3f}")
        print("    Confusion matrix:\n", confusion_matrix(y_te, y_pred, labels=[0,1]))

        fpr, tpr, _ = roc_curve(y_te, y_prob)
        ax.plot(fpr, tpr, label=f"{name} (AUC={mets['roc_auc']:.2f})")

    ax.plot([0,1],[0,1],'--',color='gray')
    ax.set_title(f"{pest} — ROC Curves"); ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate"); ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig("viz_roc_curves.png", dpi=150)
plt.show()
print("\nSaved: viz_roc_curves.png")

In [ ]:
import pandas as pd
rows = []
for (pest, model_name), mets in test_metrics.items():
    cv = cv_results[(pest, model_name)]
    rows.append({
        'Pest': pest, 'Model': model_name,
        'CV_Recall_mean': cv['recall'], 'CV_Recall_std': cv['recall_std'],
        'CV_F1_mean':     cv['f1'],     'CV_F1_std':     cv['f1_std'],
        'CV_ROC_AUC':     cv['roc_auc'],
        'Test_Accuracy':  mets['accuracy'],
        'Test_Precision': mets['precision'],
        'Test_Recall':    mets['recall'],
        'Test_F1':        mets['f1'],
        'Test_ROC_AUC':   mets['roc_auc'],
    })
comp = pd.DataFrame(rows).round(3)
print("\n=== MODEL COMPARISON (CV + TEST) ===")
print(comp.to_string(index=False))
comp.to_csv("week4_model_comparison.csv", index=False)
print("\nSaved: week4_model_comparison.csv")

In [ ]:
# Time-based split: train on earlier years, test on later years
print("\n=== TIME-BASED SPLIT (honest check for temporal autocorrelation) ===\n")

for pest, X, y, years in [('RBB', X_rbb, y_rbb, years_rbb),
                          ('WSB', X_wsb, y_wsb, years_wsb)]:
    years = years.astype(int)
    y_unique = np.sort(np.unique(years))
    split_year = y_unique[len(y_unique)//2]
    tr = years <  split_year
    te = years >= split_year
    print(f"\n{pest}: train years < {split_year} (n={tr.sum()}, pos={int(y[tr].sum())})"
          f" | test years >= {split_year} (n={te.sum()}, pos={int(y[te].sum())})")

    if y[tr].sum() == 0 or y[te].sum() == 0:
        print("  ⚠ One side has zero positives — time-based metrics unreliable.")
        continue

    scaler = StandardScaler().fit(X[tr])
    X_tr_s, X_te_s = scaler.transform(X[tr]), scaler.transform(X[te])
    for name, model in models.items():
        model.fit(X_tr_s, y[tr])
        y_pred = model.predict(X_te_s)
        y_prob = model.predict_proba(X_te_s)[:,1]
        print(f"  {name}: Recall={recall_score(y[te],y_pred,zero_division=0):.3f}"
              f"  F1={f1_score(y[te],y_pred,zero_division=0):.3f}"
              f"  AUC={roc_auc_score(y[te],y_prob) if len(set(y[te]))>1 else float('nan'):.3f}")

## Week 4 — Model Comparison & Interpretation

### Models compared
- **Baseline:** Logistic Regression (`class_weight='balanced'`, max_iter=1000)
- **Second model:** Random Forest (`n_estimators=300`, `class_weight='balanced'`,
  `min_samples_leaf=3`)
  — justified by weak linear correlations from CSST101 (max |rho| ≈ 0.20);
  we hypothesized non-linear interactions might be captured.

### Validation strategy
- **Primary:** Stratified 5-fold cross-validation (reports mean ± std across folds)
- **Secondary:** Held-out 80/20 stratified test split
- **Honest check:** Time-based split (train on years < 2021, test on years ≥ 2021)

### Results

| Pest | Model | CV Recall | CV F1 | CV ROC-AUC | Test Recall | Test F1 | Test ROC-AUC |
|------|-------|-----------|-------|------------|-------------|---------|--------------|
| RBB  | **LR** | **0.55 ± 0.29** | 0.19 ± 0.11 | **0.735** | 0.50 | 0.20 | **0.843** |
| RBB  | RF     | **0.00 ± 0.00** | 0.00 | 0.620 | **0.00** | 0.00 | 0.700 |
| WSB  | **LR** | **0.67 ± 0.19** | 0.22 ± 0.06 | **0.757** | **0.75** | 0.21 | 0.786 |
| WSB  | RF     | **0.00 ± 0.00** | 0.00 | 0.767 | **0.00** | 0.00 | 0.736 |

### Interpretation

**Logistic Regression is the better model at this sample size.** It achieves
CV ROC-AUC of 0.735 (RBB) and 0.757 (WSB) — clearly above the 0.5 no-signal
baseline, indicating **real but modest predictive signal** consistent with the
weak correlations from CSST101. Test-set ROC-AUC is even higher (0.843, 0.786),
but with only 4 positives in the test set, these values are unstable and CV is
the more reliable evidence.

**Recall is moderate and variable for LR.** CV Recall of 0.55 ± 0.29 (RBB) is
unstable — some folds find most positives, others find almost none. WSB is more
stable (0.67 ± 0.19). The test confusion matrix for WSB (`[[49 21] [1 3]]`) shows
the trade-off: high recall (3 of 4 positives caught) but 21 false positives —
a direct consequence of `class_weight='balanced'` deliberately erring toward
detecting outbreaks.

**Precision is very low (0.11–0.14) — expected, not a bug.** With a base rate
of ~5% positives, a model tuned for recall will inevitably predict some false
alarms. Whether this is acceptable depends on the cost of a missed outbreak vs
the cost of a false alarm — an important discussion for the report.

**Random Forest completely failed to detect positives** (Recall = 0.00 for both
pests in CV and test). Its confusion matrix is `[[70 0] [4 0]]` — it predicted
class 0 for every observation. Cause: with only ~15–17 positives per training
fold and a tree ensemble's greedy splitting, no split threshold isolated the
rare class before purity was achieved on the majority class. `class_weight='balanced'`
shifted class probabilities but couldn't overcome the information scarcity.
**This is a known failure mode for tree ensembles on very small imbalanced
datasets** — it is not a bug, but a valid empirical finding: at n=366 with
~5% positives, RF overfits the majority class.

**Time-based split exposes temporal autocorrelation.** For RBB, when training
on years < 2021 and testing on years ≥ 2021, LR Recall drops from 0.55 (CV) to
**0.33**, and ROC-AUC drops from 0.735 to **0.608**. This confirms that the
random stratified split **overestimates performance** — the model partially
memorizes temporal patterns that don't generalize forward. The WSB time-based
split is inconclusive (training years contain 0 positives), so it cannot be
used to evaluate WSB generalization honestly.

### Honest limitations
- **Small positive class (~20 per pest)** → all metrics have high variance;
  CV mean ± std is the reliable evidence, not single test-split values.
- **Random split is optimistic** — time-based split confirmed leakage via
  temporal autocorrelation (RBB AUC drops 0.735 → 0.608).
- **RF failure is informative** — it demonstrates that non-linear models do not
  help at this sample size; the linear baseline is preferred.
- **Low precision is a design choice** — `class_weight='balanced'` prioritizes
  recall; a deployment would need to tune the threshold based on the cost of
  false alarms vs missed outbreaks.
- **Single lot, single location (Midsayap, North Cotabato)** — no generalizability
  claim; results describe this site only.
- **Observational data** — associations and predictive patterns only; no causal
  claims.

# Week 5: Model Optimization and Error Analysis
**Optimization:** threshold tuning for LR; SMOTE comparison; RF retry with depth cap

**Error analysis:** false-negative review; overfitting check

**Final metrics:** baseline vs optimized comparison

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             precision_recall_curve, classification_report)

rbb = pd.read_csv("RBB_P94.8_binary_labels.csv")
wsb = pd.read_csv("WSB_P94.3_binary_labels.csv")
weekly = pd.read_csv("weekly_lot_a_with_moon_dataset.csv")

feature_cols_raw = ['T2M','T2M_MIN','T2M_MAX','RH2M','PRECTOTCORR',
                    'WS2M','WS2M_MAX','WS2M_MIN','WD2M','GWETTOP','Moon_Category']
rbb_m, wsb_m = rbb.copy(), wsb.copy()
for c in feature_cols_raw:
    rbb_m[c] = weekly[c].values; wsb_m[c] = weekly[c].values

def build_features(df):
    out = df.copy()
    num_cols = ['T2M','T2M_MIN','T2M_MAX','RH2M','PRECTOTCORR',
                'WS2M','WS2M_MAX','WS2M_MIN','GWETTOP']
    for c in num_cols: out[c] = out[c].fillna(out[c].median())
    cats = sorted(out['Moon_Category'].dropna().unique()); n = len(cats)
    out['moon_idx'] = out['Moon_Category'].map({c:i for i,c in enumerate(cats)}).fillna(0)
    out['moon_sin'] = np.sin(2*np.pi*out['moon_idx']/n)
    out['moon_cos'] = np.cos(2*np.pi*out['moon_idx']/n)
    wd = out['WD2M'].fillna(out['WD2M'].median())
    out['wd_sin'] = np.sin(2*np.pi*wd/360); out['wd_cos'] = np.cos(2*np.pi*wd/360)
    feats = ['T2M','T2M_MIN','T2M_MAX','RH2M','PRECTOTCORR','WS2M','WS2M_MAX',
             'WS2M_MIN','GWETTOP','moon_sin','moon_cos','wd_sin','wd_cos']
    return out[feats].values, out['Binary_Label'].values, feats

X_rbb, y_rbb, feat_names = build_features(rbb_m)
X_wsb, y_wsb, _           = build_features(wsb_m)
print("RBB:", X_rbb.shape, "pos:", int(y_rbb.sum()))
print("WSB:", X_wsb.shape, "pos:", int(y_wsb.sum()))

In [ ]:
def tune_threshold(X, y, pest, seed=42):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                              stratify=y, random_state=seed)
    scaler = StandardScaler().fit(X_tr)
    Xtr, Xte = scaler.transform(X_tr), scaler.transform(X_te)
    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=seed)
    lr.fit(Xtr, y_tr)
    prob = lr.predict_proba(Xte)[:,1]

    prec, rec, thr = precision_recall_curve(y_te, prob)
    f1s = np.divide(2*prec*rec, prec+rec, out=np.zeros_like(prec), where=(prec+rec)>0)

    # Choose best threshold by F1; also report default (0.5) and high-recall (0.3)
    best_idx = f1s.argmax()
    best_thr = thr[best_idx] if best_idx < len(thr) else 0.5

    rows = []
    for label, t in [("default_0.5", 0.5), ("high_recall_0.3", 0.3),
                     (f"best_f1_{best_thr:.2f}", best_thr)]:
        y_pred = (prob >= t).astype(int)
        rows.append({
            "Pest": pest, "Threshold": label,
            "Precision": precision_score(y_te, y_pred, zero_division=0),
            "Recall":    recall_score(y_te, y_pred, zero_division=0),
            "F1":        f1_score(y_te, y_pred, zero_division=0),
            "ROC_AUC":   roc_auc_score(y_te, prob) if len(set(y_te))>1 else np.nan,
        })

    print(f"\n=== {pest} — Threshold tuning ===")
    print(pd.DataFrame(rows).round(3).to_string(index=False))

    # Plot PR curve
    plt.figure(figsize=(6,4))
    plt.plot(rec, prec, marker='.')
    plt.axhline(0.5, color='gray', linestyle='--', alpha=0.4)
    plt.title(f"{pest} — Precision-Recall curve"); plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.tight_layout(); plt.savefig(f"viz_pr_{pest}.png", dpi=150); plt.close()
    return pd.DataFrame(rows)

tune_rbb = tune_threshold(X_rbb, y_rbb, "RBB")
tune_wsb = tune_threshold(X_wsb, y_wsb, "WSB")
print("\nSaved: viz_pr_RBB.png, viz_pr_WSB.png")

In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_SMOTE = True
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "imbalanced-learn"])
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_SMOTE = True

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['precision','recall','f1','roc_auc']

print("=== SMOTE vs class_weight='balanced' (LR, 5-fold CV) ===\n")
smote_results = []
for pest, X, y in [("RBB", X_rbb, y_rbb), ("WSB", X_wsb, y_wsb)]:
    # A: class_weight
    pipe_cw = Pipeline([('scaler', StandardScaler()),
                        ('clf', LogisticRegression(class_weight='balanced',
                                                   max_iter=1000, random_state=42))])
    # B: SMOTE
    pipe_sm = ImbPipeline([('scaler', StandardScaler()),
                           ('smote', SMOTE(random_state=42, k_neighbors=3)),
                           ('clf', LogisticRegression(max_iter=1000, random_state=42))])

    for label, pipe in [("class_weight", pipe_cw), ("SMOTE", pipe_sm)]:
        sc = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
        row = {"Pest": pest, "Strategy": label}
        for m in scoring: row[m] = sc[f"test_{m}"].mean()
        smote_results.append(row)

smote_df = pd.DataFrame(smote_results).round(3)
print(smote_df.to_string(index=False))
smote_df.to_csv("week5_smote_vs_classweight.csv", index=False)
print("\nSaved: week5_smote_vs_classweight.csv")

In [ ]:
print("=== Random Forest retry — depth cap + higher min_samples_leaf ===\n")
rf_configs = {
    "RF_default":        RandomForestClassifier(n_estimators=300,
                                                class_weight='balanced',
                                                random_state=42,
                                                min_samples_leaf=3),
    "RF_depth4":         RandomForestClassifier(n_estimators=500, max_depth=4,
                                                class_weight='balanced',
                                                random_state=42,
                                                min_samples_leaf=5),
    "RF_depth3_leaf8":   RandomForestClassifier(n_estimators=500, max_depth=3,
                                                class_weight='balanced',
                                                random_state=42,
                                                min_samples_leaf=8),
}
rf_rows = []
for pest, X, y in [("RBB", X_rbb, y_rbb), ("WSB", X_wsb, y_wsb)]:
    for label, rf in rf_configs.items():
        pipe = Pipeline([('scaler', StandardScaler()), ('clf', rf)])
        sc = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
        row = {"Pest": pest, "Model": label}
        for m in scoring: row[m] = sc[f"test_{m}"].mean()
        rf_rows.append(row)
rf_df = pd.DataFrame(rf_rows).round(3)
print(rf_df.to_string(index=False))
rf_df.to_csv("week5_rf_retry.csv", index=False)
print("\nSaved: week5_rf_retry.csv")

In [ ]:
print("=== FALSE-NEGATIVE ANALYSIS (LR, held-out test) ===\n")
fn_records = []
for pest, X, y in [("RBB", X_rbb, y_rbb), ("WSB", X_wsb, y_wsb)]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                              stratify=y, random_state=42)
    idx_te = train_test_split(np.arange(len(y)), test_size=0.2,
                              stratify=y, random_state=42)[1]
    scaler = StandardScaler().fit(X_tr)
    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    lr.fit(scaler.transform(X_tr), y_tr)
    prob = lr.predict_proba(scaler.transform(X_te))[:,1]
    pred = (prob >= 0.5).astype(int)

    # Frame of missed cases
    src = (rbb_m if pest=="RBB" else wsb_m).iloc[idx_te].reset_index(drop=True)
    missed = (y_te == 1) & (pred == 0)
    caught = (y_te == 1) & (pred == 1)
    n_miss = int(missed.sum()); n_catch = int(caught.sum())

    print(f"\n{pest}: {n_catch} caught, {n_miss} missed (of {int(y_te.sum())} positives)")
    if n_miss > 0:
        m = src[missed]
        print(f"  Missed weeks — Year range: {m['Year'].min()}–{m['Year'].max()}")
        print(f"  Missed Pest_Count values: {m['Pest_Count'].tolist()}")
        print(f"  Missed T2M: {m['T2M'].round(1).tolist()}")
        print(f"  Missed RH2M: {m['RH2M'].round(0).tolist()}")
        print(f"  Missed prob (LR): {prob[missed].round(3).tolist()}")
        fn_records.append({"Pest": pest, "n_missed": n_miss,
                           "Years": f"{m['Year'].min()}–{m['Year'].max()}",
                           "PestCounts": m['Pest_Count'].tolist()})

if fn_records:
    pd.DataFrame(fn_records).to_csv("week5_false_negatives.csv", index=False)
    print("\nSaved: week5_false_negatives.csv")

In [ ]:
print("=== FINAL SUMMARY TABLE (CV, 5-fold) ===\n")
final_rows = []
for pest, X, y in [("RBB", X_rbb, y_rbb), ("WSB", X_wsb, y_wsb)]:
    # LR baseline (class_weight)
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('clf', LogisticRegression(class_weight='balanced',
                                                max_iter=1000, random_state=42))])
    sc = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
    final_rows.append({"Pest": pest, "Model": "LR (baseline)",
                       **{m: sc[f"test_{m}"].mean() for m in scoring}})
    # LR + SMOTE
    pipe = ImbPipeline([('scaler', StandardScaler()),
                        ('smote', SMOTE(random_state=42, k_neighbors=3)),
                        ('clf', LogisticRegression(max_iter=1000, random_state=42))])
    sc = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
    final_rows.append({"Pest": pest, "Model": "LR + SMOTE",
                       **{m: sc[f"test_{m}"].mean() for m in scoring}})
    # RF depth4 (best retry)
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('clf', RandomForestClassifier(n_estimators=500, max_depth=4,
                                                    class_weight='balanced',
                                                    random_state=42,
                                                    min_samples_leaf=5))])
    sc = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
    final_rows.append({"Pest": pest, "Model": "RF depth=4",
                       **{m: sc[f"test_{m}"].mean() for m in scoring}})

final = pd.DataFrame(final_rows).round(3)
print(final.to_string(index=False))
final.to_csv("week5_final_metrics.csv", index=False)
print("\nSaved: week5_final_metrics.csv")

## Week 5 — Optimization & Error Analysis

### Optimization tested
1. **Threshold tuning** (LR): swept decision threshold on PR curve.
2. **SMOTE vs class_weight='balanced'**: compared sampling strategies with 5-fold CV.
3. **RF depth cap retry**: `max_depth=4` and `max_depth=3` to test if regularization fixes the Week 4 collapse.

### Threshold tuning — results
- **RBB:** Default 0.5 gave Recall=0.50, Precision=0.125. Lowering the threshold to **0.46** improved **both** metrics: Recall=1.00 (caught all outbreaks), Precision=0.211, F1=0.348.
- **WSB:** Default 0.5 gave Recall=0.75, Precision=0.125. Best-F1 threshold of **0.75** shifted the balance to Recall=0.50, Precision=0.167 (more conservative).
- PR curves saved: `viz_pr_RBB.png`, `viz_pr_WSB.png`.

### SMOTE vs class_weight (5-fold CV)
| Pest | Strategy | Precision | Recall | F1 | ROC-AUC |
|------|----------|-----------|--------|----|---------|
| RBB  | class_weight | 0.112 | 0.55 | 0.185 | 0.735 |
| RBB  | SMOTE | 0.124 | 0.50 | 0.197 | 0.744 |
| WSB  | class_weight | 0.135 | 0.67 | 0.224 | 0.757 |
| WSB  | SMOTE | 0.123 | 0.52 | 0.198 | 0.744 |

*Conclusion:* SMOTE offers a marginal improvement for RBB but degrades WSB performance compared to class weighting. Since the difference is negligible at this sample size, `class_weight='balanced'` remains the simpler, preferred choice.

### RF retry (depth cap)
- `RF_default` completely collapsed (Recall 0.00).
- `RF_depth4` and `RF_depth3_leaf8` partially fixed the collapse (Recall improved to 0.05–0.15).
- **Final verdict:** Even with severe depth constraints, RF is vastly inferior to LR. The best RF F1 is 0.137 (WSB), whereas LR achieves 0.224. At n=366, tree-based models cannot isolate the minority class effectively.

### False-negative analysis (LR, threshold 0.5)
- **RBB:** 2 caught, **2 missed** (of 4). Missed weeks were in 2022–2023. The predicted probabilities for the missed weeks were **0.490 and 0.459** — *extremely close to the 0.5 cutoff*.
  - *Insight:* The best-F1 threshold of **0.46** discovered in the tuning step would have caught both of these missed outbreaks.
- **WSB:** 3 caught, **1 missed** (of 4). Missed week was in 2024 (prob=0.328, count=221).
  - *Insight:* This was a harder miss, as the model was quite confident it was a background week.

### Overfitting check
- LR CV std is moderate (0.29 RBB, 0.19 WSB).
- Time-based split (Week 4) showed ROC-AUC dropping to 0.608 for RBB, confirming **temporal leakage** rather than pure overfitting.

### Final evaluation metrics (CV, 5-fold)
Best model per pest:
- **RBB:** LR + SMOTE (F1 = 0.197, ROC-AUC = 0.744, Recall = 0.50) — *Or LR baseline with threshold 0.46 (Recall 1.0, F1 0.348)*.
- **WSB:** LR baseline (F1 = 0.224, ROC-AUC = 0.757, Recall = 0.67).
- **Overall:** LR decisively outperforms RF. SMOTE offers no consistent advantage over class_weight.

### Limitations, risks, bias
- **Threshold is a deployment decision:** 0.46 is best for RBB (high recall), 0.75 is best for WSB (balanced), 0.3 guarantees catching all outbreaks but spikes false alarms.
- **Small sample (n=366)** — high metric variance across folds.
- **Temporal leakage** — random split is optimistic; time-based split exposes a performance drop.
- **Gridded weather data** — measurement error.
- **Single lot** — findings reflect local conditions only.

## Week 6 — Machine Learning Documentation

**Report**
1. Methods & Model

Machine learning models were built using Python (Scikit-Learn). Pest counts were merged positionally with meteorological features. Categorical lunar phases and wind direction were cyclically encoded using sine/cosine transformations. A stratified 80/20 train-test split and 5-fold cross-validation were used. The baseline model was Logistic Regression with class_weight='balanced' to address the ~5% positive class imbalance. Random Forest was tested as a comparison model.

2. Evaluation

Logistic Regression achieved cross-validated ROC-AUC scores of 0.735 (RBB) and 0.757 (WSB), demonstrating modest but real predictive signal. Random Forest failed entirely (Recall = 0.00), predicting only the majority class due to the small sample size and severe class imbalance. Threshold tuning for the RBB model improved the F1-score from 0.200 to 0.348 by lowering the decision threshold to 0.46, which successfully caught previously missed near-boundary outbreaks.

3. Discussion & Limitations

The modest model performance (ROC-AUC ~0.75) aligns with the weak correlations found in the CSST101 analysis. Random Forest overfitted the majority class, demonstrating that complex models are unsuitable for this sample size (n=366). Threshold tuning is critical in imbalanced ecological datasets, as the default 0.5 cutoff misses outbreaks that the model actually detects. Limitations include temporal autocorrelation (random splits are optimistic), single-lot generalization, and the absence of key agricultural features. Deployment would require tuning thresholds based on the real-world cost of false alarms versus missed outbreaks.